Following reading the paper, titled "Positioning Political Texts with Large Language Models by Asking and Averaging" (Le Mens and Gallego 2025), I will now use the approach in the paper to try and classify political positions. I will use a subset of the graph that was created in "creating_graph_database_with_all_QT_data.ipynb" to do this. 

See a prompt that was slightly adapted from the paper below:

In [ ]:
import sys
import os

import numpy as np
from pydantic import BaseModel
from openai import OpenAI

from neo4j import GraphDatabase

In [ ]:
sys_msg = {"role" : "system",
           "content" : """You will be provided with the text of a locution and its corresponding propositional content that forms part of an argument from a UK political debating TV programme. 
           Your task is to decide where does the speaker stand on the 'left' to 'right' wing scale using the speaker's locution and propositional content? 
           Provide your response as a score between 0 and 100 where 0 means 'Extremely left' and 100 means 'Extremely right'. If the text does not have political content, set the score to “NA”. 
           Output in JSON format using the following template: {'Score' : int}. 
           
           Do not include any additional context, preamble, or explanation."""
           }

In [ ]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [3]:
# Cypher for the total number of nodes in the database.
cypher = "MATCH (n) RETURN count(n)"

In [4]:
records_total_num_nodes, summary_total_num_nodes, keys_total_num_nodes = driver.execute_query(cypher)

C:\Users\Jordan\AppData\Local\Temp\ipykernel_36224\861080273.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records_total_num_nodes, summary_total_num_nodes, keys_total_num_nodes = driver.execute_query(cypher)


In [5]:
print(records_total_num_nodes)

[<Record count(n)=23300>]


In [6]:
for result in records_total_num_nodes:
    print(result["count(n)"])
    print(type(result["count(n)"]))

23300
<class 'int'>


In [7]:
# Total number of nodes in the QT30 graph
records_total_num_nodes[0]["count(n)"]

23300

In [9]:
# Choose a number of random 
random_ints = np.random.randint(low=0,high=(records_total_num_nodes[0]["count(n)"]), size=(100,))
random_ints

array([ 9777,  2746, 16870,  5126,  1991, 22213, 18493, 11994,  7013,
       17110,  8539,   474, 18312, 19134,  2051, 11842, 20378, 21868,
        6206, 20458,  5783, 19874, 13150, 19506, 19581,   257, 16382,
        2055, 12763, 12056,  5357,  3824, 10715,  7075,  7025,  7825,
        2321, 22474,  9313, 10800, 17384, 11610,  8919, 22936, 18968,
        1060, 17112,  7346, 22945, 17650, 19096,  6449,  4878,  9039,
        3481,  9901, 11391, 21759,  1629, 15268, 14627, 20861,  7721,
       19243, 22887,  6645, 20931,   427, 21792, 16761, 13543, 19480,
        7411, 23132,  8242, 10744,  5491,  1159, 12214,  1429,  8761,
       17664, 10781, 18865,  8789,  9092, 22096,   469,  1167,  7544,
        3152, 13902, 21672, 12821,   218,  6114,  4358,  6535, 18214,
       11461], dtype=int32)

In [10]:
np.shape(random_ints)

(100,)

In [12]:
parent_dir = os.path.abspath(os.path.join(os.path.dirname("D:\Post-Doc\Persuasive-Dialogue-in-LLMs\Code\persuasionDialogueSystem\models\models.py")))

In [13]:
sys.path.append(parent_dir)

In [14]:
from models import GenerateLLMResponses

In [16]:
class PoliticalPositionScore(BaseModel):
    Score: int

### GPT 3.5

In [29]:
model="gpt-35-turbo"
temp=0
top_p= 0.5
seed= 0

In [ ]:
prompt_tokens = []
completion_tokens = []

for index, sample in enumerate(random_ints):
    print(index)
    #print("qt30_"+str(sample))
    cypher = "MATCH (n) WHERE n.unique_id = '"+str(sample)+"' RETURN (n)"
    records, summary, keys = driver.execute_query(cypher)
    #print(records)
    print(records[0]["n"]._properties)
    
    human_msg = {"role" : "user",
                 "content" : """Score the speaker's political position between 0 ('Extremely left') and 100 ('Extremely right') using the following locution and proposition.
                 
                 Proposition: '""" + records[0]["n"]._properties["proposition"] + """'

                 Locution: '""" + records[0]["n"]._properties["locution"].split(":", 1)[-1].strip() + """'

                 Do not write an introduction or summary. Output in JSON format using the following template: {'Score' : int}"""}
    
    #print(human_msg)
    result, tokens = GenerateLLMResponses(model_choice=model,
                                  prompt = [sys_msg, human_msg],
                                  temperature=temp,
                                  top_p= top_p,
                                  seed= seed,
                                  datatype_schema=PoliticalPositionScore).return_completion()
    print("\n")
    print("Locution: "+records[0]["n"]._properties["locution"].split(":", 1)[-1].strip())
    print("Proposition: "+records[0]["n"]._properties["proposition"])
    print("Speaker: "+records[0]["n"]._properties["speaker"])
    print(result)
    print("\n")
    print("Prompt tokens: "+str(tokens.usage.prompt_tokens))
    prompt_tokens.append(tokens.usage.prompt_tokens)
    print("Completion tokens: " + str(tokens.usage.completion_tokens))
    completion_tokens.append(tokens.usage.completion_tokens)
    print("Total tokens: "+str(tokens.usage.total_tokens))
    print("-------")
    print("\n\n\n")


0
{'locution': "AudienceMember 20210708QT02 : that's a distraction from what's going on", 'unique_id': '9777', 'utterance_type': '___Claim___', 'aif_node_id': 'qt30_9663', 'proposition': "the acceleration of the vassal programme is a distraction from what's going on", 'speaker': 'AudienceMember 20210708QT02', 'illocutionary_force': 'Asserting'}


C:\Users\Jordan\AppData\Local\Temp\ipykernel_36224\1400874349.py:8: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query(cypher)




Locution: that's a distraction from what's going on
Proposition: the acceleration of the vassal programme is a distraction from what's going on
Speaker: AudienceMember 20210708QT02
{'Score': 50}


Prompt tokens: 247
Completion tokens: 7
Total tokens: 254
-------




1
{'locution': 'Fiona Bruce : anyone listening and looking at the graphs must be thinking it’s just a matter of time', 'unique_id': '2746', 'utterance_type': '___Claim___', 'aif_node_id': 'qt30_2728', 'proposition': 'anyone listening and looking at the graphs must be thinking it is just a matter of time', 'speaker': 'Fiona Bruce', 'illocutionary_force': 'Asserting'}


Locution: anyone listening and looking at the graphs must be thinking it’s just a matter of time
Proposition: anyone listening and looking at the graphs must be thinking it is just a matter of time
Speaker: Fiona Bruce
{'Score': 50}


Prompt tokens: 258
Completion tokens: 7
Total tokens: 265
-------




2
{'locution': 'Louise Haigh : the last five years have

In [31]:
print("Max prompt tokens: "+str(np.max(prompt_tokens)))

Max prompt tokens: 309


In [33]:
print("Max completion tokens: " +str( np.max(completion_tokens)))

Max completion tokens: 7


### GPT 4 Turbo

In [34]:
model="gpt-4o-turbo"
temp=0
top_p= 0.5
seed= 0

In [35]:
prompt_tokens = []
completion_tokens = []

for index, sample in enumerate(random_ints):
    print(index)
    #print("qt30_"+str(sample))
    cypher = "MATCH (n) WHERE n.unique_id = '"+str(sample)+"' RETURN (n)"
    records, summary, keys = driver.execute_query(cypher)
    #print(records)
    print(records[0]["n"]._properties)
    
    human_msg = {"role" : "user",
                 "content" : """Score the speaker's political position between 0 ('Extremely left') and 100 ('Extremely right') using the following locution and proposition.
                 
                 Proposition: '""" + records[0]["n"]._properties["proposition"] + """'

                 Locution: '""" + records[0]["n"]._properties["locution"].split(":", 1)[-1].strip() + """'

                 Do not write an introduction or summary. Output in JSON format using the following template: {'Score' : int}"""}
    
    #print(human_msg)
    result, tokens = GenerateLLMResponses(model_choice=model,
                                  prompt = [sys_msg, human_msg],
                                  temperature=temp,
                                  top_p= top_p,
                                  seed= seed,
                                  datatype_schema=PoliticalPositionScore).return_completion()
    print("\n")
    print("Locution: "+records[0]["n"]._properties["locution"].split(":", 1)[-1].strip())
    print("Proposition: "+records[0]["n"]._properties["proposition"])
    print("Speaker: "+records[0]["n"]._properties["speaker"])
    print(result)
    print("\n")
    print("Prompt tokens: "+str(tokens.usage.prompt_tokens))
    prompt_tokens.append(tokens.usage.prompt_tokens)
    print("Completion tokens: " + str(tokens.usage.completion_tokens))
    completion_tokens.append(tokens.usage.completion_tokens)
    print("Total tokens: "+str(tokens.usage.total_tokens))
    print("-------")
    print("\n\n\n")

0
{'locution': "AudienceMember 20210708QT02 : that's a distraction from what's going on", 'unique_id': '9777', 'utterance_type': '___Claim___', 'aif_node_id': 'qt30_9663', 'proposition': "the acceleration of the vassal programme is a distraction from what's going on", 'speaker': 'AudienceMember 20210708QT02', 'illocutionary_force': 'Asserting'}


C:\Users\Jordan\AppData\Local\Temp\ipykernel_36224\729510509.py:8: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query(cypher)




Locution: that's a distraction from what's going on
Proposition: the acceleration of the vassal programme is a distraction from what's going on
Speaker: AudienceMember 20210708QT02
{'Score': 'NA'}


Prompt tokens: 247
Completion tokens: 6
Total tokens: 253
-------




1
{'locution': 'Fiona Bruce : anyone listening and looking at the graphs must be thinking it’s just a matter of time', 'unique_id': '2746', 'utterance_type': '___Claim___', 'aif_node_id': 'qt30_2728', 'proposition': 'anyone listening and looking at the graphs must be thinking it is just a matter of time', 'speaker': 'Fiona Bruce', 'illocutionary_force': 'Asserting'}


Locution: anyone listening and looking at the graphs must be thinking it’s just a matter of time
Proposition: anyone listening and looking at the graphs must be thinking it is just a matter of time
Speaker: Fiona Bruce
{'Score': 'NA'}


Prompt tokens: 258
Completion tokens: 6
Total tokens: 264
-------




2
{'locution': 'Louise Haigh : the last five years 

In [36]:
print("Max prompt tokens: "+str(np.max(prompt_tokens)))

Max prompt tokens: 309


In [37]:
print("Max completion tokens: " +str( np.max(completion_tokens)))

Max completion tokens: 6


### GPT 4o

In [38]:
model="gpt-4o"
temp=0
top_p= 0.5
seed= 0

In [39]:
prompt_tokens = []
completion_tokens = []

for index, sample in enumerate(random_ints):
    print(index)
    #print("qt30_"+str(sample))
    cypher = "MATCH (n) WHERE n.unique_id = '"+str(sample)+"' RETURN (n)"
    records, summary, keys = driver.execute_query(cypher)
    #print(records)
    print(records[0]["n"]._properties)
    
    human_msg = {"role" : "user",
                 "content" : """Score the speaker's political position between 0 ('Extremely left') and 100 ('Extremely right') using the following locution and proposition.
                 
                 Proposition: '""" + records[0]["n"]._properties["proposition"] + """'

                 Locution: '""" + records[0]["n"]._properties["locution"].split(":", 1)[-1].strip() + """'

                 Do not write an introduction or summary. Output in JSON format using the following template: {'Score' : int}"""}
    
    #print(human_msg)
    result, tokens = GenerateLLMResponses(model_choice=model,
                                  prompt = [sys_msg, human_msg],
                                  temperature=temp,
                                  top_p= top_p,
                                  seed= seed,
                                  datatype_schema=PoliticalPositionScore).return_completion()
    print("\n")
    print("Locution: "+records[0]["n"]._properties["locution"].split(":", 1)[-1].strip())
    print("Proposition: "+records[0]["n"]._properties["proposition"])
    print("Speaker: "+records[0]["n"]._properties["speaker"])
    print(result)
    print("\n")
    print("Prompt tokens: "+str(tokens.usage.prompt_tokens))
    prompt_tokens.append(tokens.usage.prompt_tokens)
    print("Completion tokens: " + str(tokens.usage.completion_tokens))
    completion_tokens.append(tokens.usage.completion_tokens)
    print("Total tokens: "+str(tokens.usage.total_tokens))
    print("-------")
    print("\n\n\n")

0
{'locution': "AudienceMember 20210708QT02 : that's a distraction from what's going on", 'unique_id': '9777', 'utterance_type': '___Claim___', 'aif_node_id': 'qt30_9663', 'proposition': "the acceleration of the vassal programme is a distraction from what's going on", 'speaker': 'AudienceMember 20210708QT02', 'illocutionary_force': 'Asserting'}


C:\Users\Jordan\AppData\Local\Temp\ipykernel_36224\729510509.py:8: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query(cypher)




Locution: that's a distraction from what's going on
Proposition: the acceleration of the vassal programme is a distraction from what's going on
Speaker: AudienceMember 20210708QT02
{'Score': 50}


Prompt tokens: 245
Completion tokens: 6
Total tokens: 251
-------




1
{'locution': 'Fiona Bruce : anyone listening and looking at the graphs must be thinking it’s just a matter of time', 'unique_id': '2746', 'utterance_type': '___Claim___', 'aif_node_id': 'qt30_2728', 'proposition': 'anyone listening and looking at the graphs must be thinking it is just a matter of time', 'speaker': 'Fiona Bruce', 'illocutionary_force': 'Asserting'}


Locution: anyone listening and looking at the graphs must be thinking it’s just a matter of time
Proposition: anyone listening and looking at the graphs must be thinking it is just a matter of time
Speaker: Fiona Bruce
{'Score': 'NA'}


Prompt tokens: 259
Completion tokens: 6
Total tokens: 265
-------




2
{'locution': 'Louise Haigh : the last five years ha

In [40]:
print("Max prompt tokens: "+str(np.max(prompt_tokens)))

Max prompt tokens: 308


In [41]:
print("Max completion tokens: " +str( np.max(completion_tokens)))

Max completion tokens: 7
